[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/08_custom_vjp.ipynb)

# 🔴 Hard: Stable log(1+exp(x)) with custom_vjp

*JAX Fundamentals*
Implement the softplus function

$$f(x) = \log(1 + e^{x})$$

so that **both** the value and its gradient are numerically stable for large `x`.

### The problem
The naive implementation is a classic trap:

```python
def log1pexp(x):
    return jnp.log(1.0 + jnp.exp(x))

log1pexp(100.0)            # inf   — exp(100) overflows
jax.grad(log1pexp)(100.0)  # nan   — autodiff differentiates through the inf
```

The true value at `x = 100` is `100.0` (to float precision) and the true
gradient is `1.0`. Autodiff is doing exactly what you told it — the fix is to
tell it something better.

### Rules
- Decorate with `@jax.custom_vjp` and register `defvjp(fwd, bwd)`
- The **forward** pass must be stable — `logaddexp`, not `log(1 + exp(x))`
- The **backward** pass must return the analytic derivative
  $f'(x) = \sigma(x) = \frac{1}{1 + e^{-x}}$
- `bwd` must return a **tuple**, one entry per primal input
- Must remain `jit`-able and `vmap`-able

### Why it matters
`custom_vjp` is how you rescue a gradient that is mathematically fine but
numerically catastrophic — the same technique behind stable `logsumexp`, the
straight-through estimator in quantization, gradient reversal layers, and
implicit differentiation of solvers. Knowing *when* autodiff needs help, and how
to hand it the analytic answer, is a strong senior-level signal.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


@jax.custom_vjp
def log1pexp(x):
    """Numerically stable log(1 + exp(x))."""
    pass  # Replace this


def log1pexp_fwd(x):
    """Return (output, residuals_needed_by_bwd)."""
    pass  # Replace this


def log1pexp_bwd(res, g):
    """Return a TUPLE of gradients, one per primal input."""
    pass  # Replace this


log1pexp.defvjp(log1pexp_fwd, log1pexp_bwd)

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

naive = lambda x: jnp.log(1.0 + jnp.exp(x))

for x in [0.0, 10.0, 100.0]:
    print(f"x={x:>6}  yours={log1pexp(x):>10.4f}  naive={naive(x):>10.4f}")

print()
for x in [0.0, 100.0]:
    print(f"x={x:>6}  grad yours={jax.grad(log1pexp)(x):.6f}  "
          f"grad naive={jax.grad(naive)(x):.6f}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("custom_vjp")

# hint("custom_vjp")      # stuck? nudge without the answer
# solution("custom_vjp")  # spoiler: the reference implementation